In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

index.fit(chunked_docs)

In [2]:
class SearchIndexTools:

    def __init__(self, index):
        self.index = index

    def search(self, query):
        return self.index.search(
            query=query,
            num_results=5
        )

    def add_entry(self, filename, title, description, content):
        entry = {
            "start": 0,
            "content": content,
            "title": title,
            "description": description,
            "filename": filename,
        }

        self.index.append(entry)

        return "OK"

In [3]:
search_tools = SearchIndexTools(index)

search_tools.search("dashboard")

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the

In [4]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'add_entry',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title': {'type': 'string', 'description': 'title parameter'},
    'description': {'type': 'string', 'description': 'description parameter'},
    'content': {'type': 'string', 'description': 'content parameter'}},
   'required': ['filename', 'title', 'description', 'content'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [5]:
from typing import Any

class SearchIndexTools:

    def __init__(self, index: Any):
        self.index = index

    def search(self, query: str):
        """
        Search the documentation database for relevant results.

        Args:
            query: The search query to look up in the index.
        """
        return self.index.search(
            query=query,
            num_results=5
        )

    def add_entry(
        self,
        filename: str,
        title: str,
        description: str,
        content: str
    ):
        """
        Add a new documentation entry to the index.

        Args:
            filename: The source filename associated with the entry.
            title: The title of the documentation entry.
            description: A short description summarizing the entry.
            content: The full content of the documentation entry.
        """
        entry = {
            "start": 0,
            "content": content,
            "title": title,
            "description": description,
            "filename": filename,
        }

        self.index.append(entry)

        return "OK"

In [6]:
search_tools = SearchIndexTools(index)

agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename: The source filename associated with the entry.\n    title: The title of the documentation entry.\n    description: A short description summarizing the entry.\n    content: The full content of the documentation entry.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title': {'type': 'string', 'description': 'title parameter'},
    'description': {'type': 'string', 'description': 'description parameter'},
    'content': {'type': 'string', 'description': 'content parameter'}},
   'required': ['filename', 'title', 'description', 'content'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results.\n\nArgs:\n    query: The search query to look up in the index.',
  'parameters': {'type':

In [8]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to
include the word 'evidently' in search results
"""

In [9]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm_client
)

In [10]:
result = agent.loop("how can I create evidently dashboard?")

In [11]:
print(result.last_message)

To create a dashboard, you'll need to follow these key steps:

### Step 1: Understand the Dashboard Structure
A dashboard serves as a centralized view for tracking the performance of your AI applications, allowing you to visualize evaluation results over time. Each project has its own dashboard, which is initially empty. To populate it, you must first run evaluations and save at least one report within the project. 

### Step 2: Adding Tabs and Panels
You can logically organize the dashboard using tabs, which can be added as follows:
- Enter "Edit" mode on the dashboard (top right corner).
- Click on the plus sign to add a new tab.
- Alternatively, tabs can also be created while adding panels using the Python API: 
```python
project.dashboard.add_tab("New Tab Name")
```

Next, you can add various panels to visualize the data. Panels may include counters, line plots, pie charts, etc. Here’s how to add a panel using the Python API:
```python
project.dashboard.add_panel(
    DashboardPane

In [12]:
import search_tools

search_tools_object = search_tools.SearchIndexTools(index)

In [13]:
search_tools_object.search("dashboard")

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the

In [14]:
agent_tools = Tools()
agent_tools.add_tools(search_tools_object)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename: The source filename associated with the entry.\n    title: The title of the documentation entry.\n    description: A short description summarizing the entry.\n    content: The full content of the documentation entry.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title': {'type': 'string', 'description': 'title parameter'},
    'description': {'type': 'string', 'description': 'description parameter'},
    'content': {'type': 'string', 'description': 'content parameter'}},
   'required': ['filename', 'title', 'description', 'content'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results.\n\nArgs:\n    query: The search query to look up in the index.',
  'parameters': {'type':